# OLD TEST mở rộng: UCB-VAE, Epsilon-Greedy và Buy-and-Hold

Notebook này giữ lại ba biểu đồ đối chứng của `old_test.ipynb`, thay hai mô hình học tăng cường bằng cấu hình cải tiến đã dùng trong `UCB009_vs_EpsilonGreedy_20seed.ipynb`, và bổ sung hai biểu đồ so sánh cả ba chiến lược.

## Cách chạy trên 6 máy Kaggle

Mỗi máy chỉ đổi `TICKER` trong cell cấu hình. Sáu mã gợi ý là `ACB`, `FPT`, `GAS`, `HPG`, `SSI`, `VCB`; notebook cũng chấp nhận `MSN` và `VNM` nếu bạn muốn thay một mã. Mỗi máy tự chạy cả hai giai đoạn `GOOD` và `BAD`, hai mô hình RL trên 20 seed `41..60`, cùng một Buy-and-Hold xác định.

- UCB-VAE: `vae_019 + ucb_009`, có cost network, robust loss, reward shaping và confidence-weighted risk penalty.
- Epsilon-Greedy: cấu hình Battle, robust loss và reward shaping.
- Chuẩn hóa: frozen scaler được fit riêng cho từng mã/giai đoạn và **chỉ từ train**; hai mô hình dùng chung đúng scaler đó.
- Resume: CSV được ghi sau mỗi seed; chạy lại với `RESUME=True` sẽ bỏ qua seed đã hoàn tất. Chỉ một checkpoint XAI được giữ cho mỗi máy.

Các biểu đồ cũ được giữ nguyên:

1. Learning Curve Comparison — UCB-VAE và Epsilon-Greedy.
2. Portfolio Value Comparison — UCB-VAE và Epsilon-Greedy.
3. Final Profit Across Runs — UCB-VAE và Epsilon-Greedy.

Hai biểu đồ mới:

4. Portfolio Value Comparison — cả ba chiến lược.
5. Final Test Metrics — cả ba chiến lược.


In [ ]:
!if [ ! -d SARSA_FinancialRL ]; then git clone https://github.com/kohi-vip/SARSA_FinancialRL.git; else echo "SARSA_FinancialRL already exists"; fi


In [ ]:
!pip install -q numpy pandas matplotlib tqdm torch tabulate


In [ ]:
# CELL 1 — Cấu hình một máy / một mã cổ phiếu
from __future__ import annotations

import gc, json, os, random, time, traceback
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Mapping, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from tqdm.auto import tqdm

# Mỗi máy đổi đúng biến này. Sáu máy gợi ý: ACB, FPT, GAS, HPG, SSI, VCB.
TICKER = "HPG"
ALLOWED_TICKERS = ("ACB", "FPT", "GAS", "HPG", "SSI", "VCB", "MSN", "VNM")
PERIODS = ("GOOD", "BAD")
MODEL_KEYS = ("UNCERTAINTY_AWARE_UCB_009", "EPSILON_GREEDY")
SEEDS = tuple(range(41, 61))  # 20 seed: 41..60

RUN_TRAINING = True
RESUME = True
SAVE_XAI_CHECKPOINT = True
XAI_MODEL_KEY = "UNCERTAINTY_AWARE_UCB_009"
XAI_PERIOD = "BAD"  # Đổi thành GOOD nếu cần giải thích mô hình ở giai đoạn tốt.
XAI_SELECTION_METRIC = "train_sharpe"  # Không dùng test để chọn checkpoint.

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GAMMA, NN_LR, COST_LR, EPISODES = 0.95, 5e-5, 1e-3, 45
SARSA_ALPHA, Q_WEIGHT_DECAY, BATCH_SIZE = 0.60, 1e-4, 128
LATENT_DIM, BALANCE_INIT, TRANSACTION_FEE = 16, 1_000.0, 0.001
W_RISK, W_STABILITY, ZETA = 0.15, 0.05, 0.05
ACTION_VALUES = np.arange(-5, 6, dtype=np.int64)
RISK_FREE_RATE_PERCENT, SCALER_SEED = 2.0, 43
CURRENT_RUN_SEED = SEEDS[0]

LOCKED_VAE_019 = {
    "vae_latent_dim": 16, "vae_lr": 1e-4, "vae_beta_kl": 5e-2,
    "vae_batch_size": 256, "bootstrap_trajectories": 5,
    "bootstrap_updates": 100, "online_aux_updates": 1,
    "vae_replay_capacity": 50_000,
}
MODEL_CONFIGS = {
    "UNCERTAINTY_AWARE_UCB_009": {
        "model_key": "UNCERTAINTY_AWARE_UCB_009",
        "label": "Uncertainty-Aware UCB-VAE — vae_019 + ucb_009",
        "strategy": "ucb", "use_cost": True, "robust_loss": True,
        "weight_decay": Q_WEIGHT_DECAY, "reward_shaping": True,
        "kl_reduction": "sum", "beta_0": 0.03,
        "beta_decay": 0.90, "beta_min": 0.01, **LOCKED_VAE_019,
    },
    "EPSILON_GREEDY": {
        "model_key": "EPSILON_GREEDY",
        "label": "Epsilon-Greedy — Battle strategy",
        "strategy": "epsilon", "use_cost": False, "robust_loss": True,
        "weight_decay": 0.0, "reward_shaping": True,
        "epsilon_init": 1.0, "epsilon_decay": 0.95, "epsilon_min": 0.05,
    },
}

if TICKER not in ALLOWED_TICKERS:
    raise ValueError(f"TICKER phải thuộc {ALLOWED_TICKERS}.")
if len(SEEDS) != 20 or SEEDS[0] != 41 or SEEDS[-1] != 60:
    raise ValueError("SEEDS phải là đúng 20 seed từ 41 đến 60.")

def set_seed(seed: int) -> None:
    global CURRENT_RUN_SEED
    CURRENT_RUN_SEED = int(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    try: torch.use_deterministic_algorithms(True, warn_only=True)
    except TypeError: torch.use_deterministic_algorithms(True)

set_seed(SEEDS[0])
print({"ticker": TICKER, "periods": PERIODS, "models": MODEL_KEYS,
       "seeds": SEEDS, "device": str(DEVICE)})


In [ ]:
# CELL 2 — Nạp dữ liệu GOOD/BAD, môi trường cải tiến và frozen scaler train-only
def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd / "SARSA_FinancialRL", Path("/kaggle/working/SARSA_FinancialRL"), *cwd.parents]
    for candidate in candidates:
        if (candidate / "data" / "data_storer" / "data_research").exists():
            return candidate
    raise FileNotFoundError("Không tìm thấy project root chứa data/data_storer/data_research.")

PROJECT_ROOT = find_project_root()
DATA_ROOT = PROJECT_ROOT / "data" / "data_storer" / "data_research"
OUTPUT_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else PROJECT_ROOT / "kaggle_working"
RUN_ROOT = OUTPUT_ROOT / "old_test_three_strategy" / TICKER
RUN_ROOT.mkdir(parents=True, exist_ok=True)
REQUIRED_COLUMNS = ["time", "close", "MACD", "RSI", "CCI", "ADX"]

def load_period_data(ticker: str, period: str) -> Tuple[pd.DataFrame, pd.DataFrame]:
    prefix = period.lower()
    train_path = DATA_ROOT / "train" / f"{prefix}_train_{ticker}.csv"
    test_path = DATA_ROOT / "test" / f"{prefix}_test_{ticker}.csv"
    if not train_path.exists() or not test_path.exists():
        raise FileNotFoundError(f"Thiếu dữ liệu {ticker} {period}: {train_path} hoặc {test_path}")
    train, test = pd.read_csv(train_path), pd.read_csv(test_path)
    for label, frame in (("train", train), ("test", test)):
        missing = sorted(set(REQUIRED_COLUMNS) - set(frame.columns))
        if missing: raise ValueError(f"{ticker} {period} {label} thiếu cột: {missing}")
        frame["time"] = pd.to_datetime(frame["time"], errors="raise")
        frame[REQUIRED_COLUMNS[1:]] = frame[REQUIRED_COLUMNS[1:]].apply(pd.to_numeric, errors="coerce")
        if frame[REQUIRED_COLUMNS[1:]].isna().any().any():
            raise ValueError(f"{ticker} {period} {label} có NaN hoặc giá trị không hợp lệ.")
    train, test = train.sort_values("time"), test.sort_values("time")
    if train["time"].max() >= test["time"].min():
        raise ValueError(f"Train/Test {ticker} {period} chồng lấn thời gian.")
    return train.reset_index(drop=True), test.reset_index(drop=True)

def raw_state(row: pd.Series, cash: float, position: int) -> np.ndarray:
    return np.asarray([row["close"], cash, position, row["MACD"], row["RSI"], row["CCI"], row["ADX"]], dtype=np.float32)

class TradingEnv:
    """Môi trường cải tiến giữ nguyên từ notebook tìm tham số/Top-5 UCB."""
    def __init__(self, data: pd.DataFrame, reward_shaping: bool, initial_cash: float = BALANCE_INIT):
        if len(data) < 2: raise ValueError("TradingEnv cần ít nhất hai quan sát.")
        self.data = data.reset_index(drop=True); self.reward_shaping = bool(reward_shaping)
        self.initial_cash = float(initial_cash); self.reset()
    def reset(self) -> np.ndarray:
        self.index, self.cash, self.position = 0, self.initial_cash, 0
        self.peak_value = self.initial_cash; self.portfolio_history = [self.initial_cash]; self.var_targets = []
        return self._state()
    def _state(self) -> np.ndarray:
        return raw_state(self.data.iloc[self.index], self.cash, self.position)
    def _execute(self, requested: int, price: float) -> int:
        if requested > 0:
            executed = min(int(requested), int(self.cash // (price * (1.0 + TRANSACTION_FEE))))
        else:
            executed = -min(-int(requested), self.position)
        traded_value = abs(executed) * price
        self.cash -= executed * price + traded_value * TRANSACTION_FEE
        self.position += executed
        return executed
    def step(self, action: int):
        current_price = float(self.data.iloc[self.index]["close"])
        previous_value = self.cash + self.position * current_price
        executed = self._execute(int(action), current_price)
        self.index += 1
        next_price = float(self.data.iloc[self.index]["close"])
        portfolio_value = self.cash + self.position * next_price
        raw_profit = portfolio_value - previous_value
        portfolio_return = raw_profit / max(abs(previous_value), 1e-8)
        var_target = max(-portfolio_return, 0.0)
        self.peak_value = max(self.peak_value, portfolio_value)
        drawdown = (self.peak_value - portfolio_value) / max(self.peak_value, 1e-8)
        asset_return = next_price / max(current_price, 1e-8) - 1.0
        reward = raw_profit
        if self.reward_shaping:
            reward -= W_RISK * abs(drawdown) + W_STABILITY * abs(asset_return)
        self.portfolio_history.append(float(portfolio_value)); self.var_targets.append(float(var_target))
        done = self.index >= len(self.data) - 1
        info = {"raw_profit": float(raw_profit), "portfolio_return": float(portfolio_return),
                "var_target": float(var_target), "drawdown": float(drawdown),
                "executed_action": int(executed), "portfolio_value": float(portfolio_value)}
        return self._state(), float(reward), done, info

@dataclass(frozen=True)
class FrozenScaler:
    mean: np.ndarray
    std: np.ndarray
    def transform(self, states: np.ndarray) -> np.ndarray:
        return (np.asarray(states, dtype=np.float32) - self.mean) / self.std

def calibration_states(train: pd.DataFrame, trajectories: int = 5) -> np.ndarray:
    rng = np.random.default_rng(SCALER_SEED); states = []
    for _ in range(trajectories):
        env = TradingEnv(train, reward_shaping=False); state, done = env.reset(), False
        while not done:
            states.append(state.copy())
            state, _, done, _ = env.step(int(rng.choice(ACTION_VALUES)))
    return np.asarray(states, dtype=np.float32)

def fit_frozen_scaler(train: pd.DataFrame) -> FrozenScaler:
    calibration = calibration_states(train)
    mean = calibration.mean(axis=0, dtype=np.float64).astype(np.float32)
    std = np.maximum(calibration.std(axis=0, dtype=np.float64), 1e-6).astype(np.float32)
    scaler = FrozenScaler(mean.copy(), std.copy())
    scaler.mean.setflags(write=False); scaler.std.setflags(write=False)
    return scaler

for _period in PERIODS:
    _train, _test = load_period_data(TICKER, _period)
    print(_period, {"train": (str(_train.time.min().date()), str(_train.time.max().date()), len(_train)),
                    "test": (str(_test.time.min().date()), str(_test.time.max().date()), len(_test))})
print("Output:", RUN_ROOT)


In [ ]:
# CELL 3 — Q-network, EpistemicVAE vae_019 và Cost Network
class QNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(7, 32), nn.ReLU(), nn.Linear(32, 11))
    def forward(self, states: torch.Tensor) -> torch.Tensor:
        return self.net(states)

class EpistemicVAE(nn.Module):
    def __init__(self, latent_dim: int = LATENT_DIM):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(18, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU())
        self.fc_mu = nn.Linear(32, latent_dim); self.fc_logvar = nn.Linear(32, latent_dim)
        self.decoder = nn.Sequential(nn.Linear(latent_dim, 32), nn.ReLU(), nn.Linear(32, 64), nn.ReLU(), nn.Linear(64, 18))
    def encode(self, x: torch.Tensor):
        hidden = self.encoder(x); return self.fc_mu(hidden), self.fc_logvar(hidden)
    def forward(self, states: torch.Tensor, actions_onehot: torch.Tensor):
        x = torch.cat([states, actions_onehot], dim=-1)
        mu, logvar = self.encode(x); std = torch.exp(0.5 * logvar)
        return self.decoder(mu + torch.randn_like(std) * std), mu, logvar
    def compute_u_ep(self, states, actions_onehot, kl_reduction: str = "sum"):
        x = torch.cat([states, actions_onehot], dim=-1)
        mu, logvar = self.encode(x)
        terms = -0.5 * (1.0 + logvar - mu.square() - logvar.exp())
        if kl_reduction == "sum": kl = torch.sum(terms, dim=-1)
        elif kl_reduction == "mean": kl = torch.mean(terms, dim=-1)
        else: raise ValueError("kl_reduction phải là 'sum' hoặc 'mean'.")
        reconstructed_95 = self.decoder(mu + 1.96 * torch.exp(0.5 * logvar))
        return kl + torch.norm(x - reconstructed_95, p=2, dim=-1)

class CostNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(18, 32), nn.ReLU(), nn.Linear(32, 1), nn.Sigmoid())
    def forward(self, states: torch.Tensor, actions_onehot: torch.Tensor) -> torch.Tensor:
        return self.net(torch.cat([states, actions_onehot], dim=-1)).squeeze(-1)

def onehot(action_indices: torch.Tensor) -> torch.Tensor:
    return F.one_hot(action_indices.long(), num_classes=11).to(dtype=torch.float32)


In [ ]:
# CELL 4 — Replay, loss và confidence-weighted risk penalty
def reconstruction_loss(prediction, target, robust: bool):
    return F.huber_loss(prediction, target) if robust else F.mse_loss(prediction, target)

def vae_loss(reconstructed, target, mu, logvar, beta_kl: float, robust: bool):
    recon = reconstruction_loss(reconstructed, target, robust)
    kl = -0.5 * torch.mean(1.0 + logvar - mu.square() - logvar.exp())
    return recon + float(beta_kl) * kl, recon, kl

class ReplayBuffer:
    def __init__(self, capacity: int):
        self.capacity = int(capacity); self.states = []; self.actions = []; self.var_targets = []
    def add(self, states, action_indices, var_targets):
        for state, action, var_target in zip(states, action_indices, var_targets):
            self.states.append(np.asarray(state, dtype=np.float32)); self.actions.append(int(action)); self.var_targets.append(float(var_target))
        overflow = len(self.states) - self.capacity
        if overflow > 0:
            del self.states[:overflow]; del self.actions[:overflow]; del self.var_targets[:overflow]
    def sample(self, batch_size: int):
        size = min(int(batch_size), len(self.states))
        if size == 0: raise RuntimeError("Không thể sample replay buffer rỗng.")
        idx = np.random.choice(len(self.states), size=size, replace=False)
        return (np.asarray([self.states[i] for i in idx]), np.asarray([self.actions[i] for i in idx]),
                np.asarray([self.var_targets[i] for i in idx], dtype=np.float32))
    def __len__(self): return len(self.states)

def action_scores(q_network, vae, cost_network, state, beta, config, scaler, collect_details=False):
    scaled = scaler.transform(np.asarray(state).reshape(1, -1))
    state_tensor = torch.as_tensor(scaled, dtype=torch.float32, device=DEVICE)
    state_batch = state_tensor.repeat(11, 1); action_batch = torch.eye(11, dtype=torch.float32, device=DEVICE)
    q_network.eval(); vae.eval(); cost_network.eval()
    with torch.no_grad():
        q_values = q_network(state_tensor).squeeze(0)
        novelty_raw = vae.compute_u_ep(state_batch, action_batch, config["kl_reduction"])
        novelty_scaled = novelty_raw / (novelty_raw.max() + 1e-8)
        predicted_var = cost_network(state_batch, action_batch)
        confidence = 1.0 / (1.0 + novelty_raw.clamp_min(0.0))
        penalty = torch.where(predicted_var < ZETA, torch.zeros_like(predicted_var), confidence * predicted_var)
        scores = q_values - penalty + float(beta) * novelty_scaled
        action_index = int(torch.argmax(scores).item())
    details = None if not collect_details else {
        "novelty": novelty_raw.detach().cpu().numpy(),
        "predicted_var": predicted_var.detach().cpu().numpy(),
        "penalty": penalty.detach().cpu().numpy(),
    }
    return int(ACTION_VALUES[action_index]), action_index, details

def auxiliary_update(vae, vae_optimizer, cost_network, cost_optimizer, replay, config, scaler):
    raw_states, action_indices, var_targets = replay.sample(config["vae_batch_size"])
    states = torch.as_tensor(scaler.transform(raw_states), dtype=torch.float32, device=DEVICE)
    indices = torch.as_tensor(action_indices, dtype=torch.long, device=DEVICE); actions = onehot(indices)
    reconstructed, mu, logvar = vae(states, actions); target = torch.cat([states, actions], dim=-1)
    loss_vae, recon, kl = vae_loss(reconstructed, target, mu, logvar, config["vae_beta_kl"], config["robust_loss"])
    vae_optimizer.zero_grad(set_to_none=True); loss_vae.backward()
    torch.nn.utils.clip_grad_norm_(vae.parameters(), 5.0); vae_optimizer.step()
    targets = torch.as_tensor(var_targets, dtype=torch.float32, device=DEVICE)
    predicted = cost_network(states, actions); loss_cost = F.huber_loss(predicted, targets)
    cost_optimizer.zero_grad(set_to_none=True); loss_cost.backward()
    torch.nn.utils.clip_grad_norm_(cost_network.parameters(), 5.0); cost_optimizer.step()
    return float(loss_vae.detach().cpu()), float(recon.detach().cpu()), float(kl.detach().cpu()), float(loss_cost.detach().cpu())

def random_bootstrap(replay: ReplayBuffer, config, train: pd.DataFrame):
    rng = np.random.default_rng(CURRENT_RUN_SEED)
    for _ in range(int(config["bootstrap_trajectories"])):
        env = TradingEnv(train, reward_shaping=False); state, done = env.reset(), False
        states, actions, var_targets = [], [], []
        while not done:
            action_index = int(rng.integers(0, 11)); states.append(state.copy()); actions.append(action_index)
            state, _, done, info = env.step(int(ACTION_VALUES[action_index])); var_targets.append(info["var_target"])
        replay.add(states, actions, var_targets)

def collect_ucb_episode(env, q_network, vae, cost_network, beta, config, scaler):
    state, done = env.reset(), False
    states, next_states, rewards, actions, dones, var_targets = [], [], [], [], [], []
    while not done:
        action, action_index, _ = action_scores(q_network, vae, cost_network, state, beta, config, scaler)
        next_state, reward, done, info = env.step(action)
        states.append(state.copy()); next_states.append(next_state.copy()); rewards.append(reward)
        actions.append(action_index); dones.append(done); var_targets.append(info["var_target"]); state = next_state
    next_actions = actions[1:] + [actions[-1]]
    return states, next_states, rewards, actions, next_actions, dones, var_targets

def q_update(q_network, optimizer, trajectory, robust: bool, scaler):
    states, next_states, rewards, actions, next_actions, dones, _ = trajectory
    s = torch.as_tensor(scaler.transform(states), dtype=torch.float32, device=DEVICE)
    sn = torch.as_tensor(scaler.transform(next_states), dtype=torch.float32, device=DEVICE)
    r = torch.as_tensor(rewards, dtype=torch.float32, device=DEVICE)
    a = torch.as_tensor(actions, dtype=torch.long, device=DEVICE)
    an = torch.as_tensor(next_actions, dtype=torch.long, device=DEVICE)
    terminal = torch.as_tensor(dones, dtype=torch.float32, device=DEVICE); losses = []; q_network.train()
    for start in range(0, len(states), BATCH_SIZE):
        sl = slice(start, min(start + BATCH_SIZE, len(states)))
        current = q_network(s[sl]).gather(1, a[sl, None]).squeeze(1)
        with torch.no_grad():
            following = q_network(sn[sl]).gather(1, an[sl, None]).squeeze(1)
            td_target = r[sl] + GAMMA * (1.0 - terminal[sl]) * following
            target = (1.0 - SARSA_ALPHA) * current.detach() + SARSA_ALPHA * td_target
        loss = F.huber_loss(current, target) if robust else F.mse_loss(current, target)
        optimizer.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(q_network.parameters(), 5.0); optimizer.step()
        losses.append(float(loss.detach().cpu()))
    return losses


In [ ]:
# CELL 5 — Huấn luyện, đánh giá, CSV từng seed và một checkpoint dành cho XAI
def evaluation_frame(data: pd.DataFrame, previous_row: Optional[pd.Series] = None) -> pd.DataFrame:
    return data if previous_row is None else pd.concat([previous_row.to_frame().T, data], ignore_index=True)

def epsilon_action(q_network, state, epsilon: float, explore: bool, scaler):
    if explore and np.random.random() < float(epsilon):
        index = int(np.random.randint(0, len(ACTION_VALUES)))
    else:
        scaled = scaler.transform(np.asarray(state).reshape(1, -1))
        tensor = torch.as_tensor(scaled, dtype=torch.float32, device=DEVICE)
        q_network.eval()
        with torch.no_grad(): index = int(torch.argmax(q_network(tensor).squeeze(0)).item())
    return int(ACTION_VALUES[index]), index

def collect_epsilon_episode(env, q_network, epsilon: float, scaler):
    state, done = env.reset(), False
    states, next_states, rewards, actions, dones, var_targets = [], [], [], [], [], []
    while not done:
        action, action_index = epsilon_action(q_network, state, epsilon, True, scaler)
        next_state, reward, done, info = env.step(action)
        states.append(state.copy()); next_states.append(next_state.copy()); rewards.append(reward)
        actions.append(action_index); dones.append(done); var_targets.append(info["var_target"]); state = next_state
    next_actions = actions[1:] + [actions[-1]]
    return states, next_states, rewards, actions, next_actions, dones, var_targets

def evaluate_strategy(q_network, vae, cost_network, control, config, scaler, data, previous_row=None):
    frame = evaluation_frame(data, previous_row)
    env = TradingEnv(frame, reward_shaping=config["reward_shaping"])
    state, done, novelty_values = env.reset(), False, []
    while not done:
        if config["strategy"] == "epsilon":
            action, _ = epsilon_action(q_network, state, 0.0, False, scaler)
        else:
            action, _, details = action_scores(q_network, vae, cost_network, state, control, config, scaler, True)
            novelty_values.extend(details["novelty"].tolist())
        state, _, done, _ = env.step(action)
    return (np.asarray(env.portfolio_history, dtype=np.float64),
            np.asarray(env.var_targets, dtype=np.float64),
            np.asarray(novelty_values, dtype=np.float64))

def period_metrics(portfolio: np.ndarray, dates: Sequence[pd.Timestamp], var_targets=None) -> Dict[str, float]:
    profit = float(portfolio[-1] - portfolio[0])
    roi = float(profit / max(abs(portfolio[0]), 1e-8) * 100.0)
    dates = pd.Series(dates).reset_index(drop=True)
    days = max((pd.Timestamp(dates.iloc[-1]) - pd.Timestamp(dates.iloc[0])).days, 1)
    ratio = float(portfolio[-1] / max(portfolio[0], 1e-8))
    arr = float((ratio ** (365.25 / days) - 1.0) * 100.0) if ratio > 0 else -100.0
    returns = np.diff(portfolio) / np.maximum(np.abs(portfolio[:-1]), 1e-8)
    annual_return = np.mean(returns) * 252.0 * 100.0 if len(returns) else 0.0
    volatility = np.std(returns) * np.sqrt(252.0) * 100.0 if len(returns) else 0.0
    sharpe = 0.0 if volatility < 1e-12 else float((annual_return - RISK_FREE_RATE_PERCENT) / volatility)
    peaks = np.maximum.accumulate(portfolio)
    mdd = float(abs(np.min((portfolio - peaks) / np.maximum(peaks, 1e-8))) * 100.0)
    violations = int(np.sum(np.asarray(var_targets) > ZETA)) if var_targets is not None else 0
    return {"profit": profit, "roi": roi, "arr": arr, "volatility": volatility,
            "sharpe": sharpe, "max_drawdown": mdd, "violations": violations}

def buy_and_hold(train: pd.DataFrame, test: pd.DataFrame):
    boundary_price = float(train.iloc[-1]["close"])
    shares = int(BALANCE_INIT // (boundary_price * (1.0 + TRANSACTION_FEE)))
    cash = BALANCE_INIT - shares * boundary_price * (1.0 + TRANSACTION_FEE)
    portfolio = [BALANCE_INIT]
    for price in test["close"].astype(float): portfolio.append(float(cash + shares * price))
    portfolio = np.asarray(portfolio, dtype=np.float64)
    returns = np.diff(portfolio) / np.maximum(np.abs(portfolio[:-1]), 1e-8)
    var_targets = np.maximum(-returns, 0.0)
    return portfolio, var_targets, shares, cash

def append_frame(path: Path, frame: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, mode="a", header=not path.exists(), index=False)

def atomic_torch_save(payload: Dict[str, Any], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    torch.save(payload, temporary); os.replace(temporary, path)

def maybe_save_xai_checkpoint(checkpoint: Dict[str, Any], metric: Dict[str, Any],
                              metrics_path: Path, model_key: str, period: str) -> bool:
    if not SAVE_XAI_CHECKPOINT or model_key != XAI_MODEL_KEY or period != XAI_PERIOD:
        return False
    score = float(metric[XAI_SELECTION_METRIC])
    previous_best = -np.inf
    if metrics_path.exists():
        previous = pd.read_csv(metrics_path)
        previous = previous[(previous["model_key"] == model_key) & (previous["period"] == period)]
        if not previous.empty and XAI_SELECTION_METRIC in previous.columns:
            previous_best = float(previous[XAI_SELECTION_METRIC].max())
    xai_path = RUN_ROOT / "xai_model_checkpoint.pt"
    if score > previous_best or not xai_path.exists():
        checkpoint["metric"] = metric
        checkpoint["xai_selection_metric"] = XAI_SELECTION_METRIC
        checkpoint["xai_selection_score"] = score
        atomic_torch_save(checkpoint, xai_path)
        print(f"[XAI checkpoint] {xai_path} | seed={metric['seed']} | {XAI_SELECTION_METRIC}={score:.6f}")
        return True
    return False

def run_one_seed(ticker, period, model_key, train, test, scaler, progress_bar=None):
    set_seed(CURRENT_RUN_SEED); config = dict(MODEL_CONFIGS[model_key])
    q_network = QNetwork().to(DEVICE)
    vae = EpistemicVAE(latent_dim=LATENT_DIM).to(DEVICE)
    cost_network = CostNetwork().to(DEVICE)
    q_optimizer = torch.optim.Adam(q_network.parameters(), lr=NN_LR, weight_decay=config["weight_decay"])
    vae_optimizer = torch.optim.Adam(vae.parameters(), lr=config["vae_lr"]) if config["strategy"] == "ucb" else None
    cost_optimizer = torch.optim.Adam(cost_network.parameters(), lr=COST_LR) if config.get("use_cost", False) else None
    replay = ReplayBuffer(config["vae_replay_capacity"]) if config["strategy"] == "ucb" else None
    q_losses, vae_losses, cost_losses, curve_rows = [], [], [], []

    if config["strategy"] == "ucb":
        random_bootstrap(replay, config, train)
        if progress_bar is not None: progress_bar.set_postfix(period=period, model="UCB", seed=CURRENT_RUN_SEED, phase="warmup")
        for _ in range(int(config["bootstrap_updates"])):
            lv, _, _, lc = auxiliary_update(vae, vae_optimizer, cost_network, cost_optimizer, replay, config, scaler)
            vae_losses.append(lv); cost_losses.append(lc)

    for episode in range(EPISODES):
        if config["strategy"] == "epsilon":
            control = max(float(config["epsilon_min"]), float(config["epsilon_init"]) * float(config["epsilon_decay"]) ** episode)
            trajectory = collect_epsilon_episode(TradingEnv(train, config["reward_shaping"]), q_network, control, scaler)
        else:
            control = max(float(config["beta_min"]), float(config["beta_0"]) * float(config["beta_decay"]) ** episode)
            trajectory = collect_ucb_episode(TradingEnv(train, config["reward_shaping"]), q_network, vae, cost_network, control, config, scaler)
            replay.add(trajectory[0], trajectory[3], trajectory[6])
        q_losses.extend(q_update(q_network, q_optimizer, trajectory, config["robust_loss"], scaler))
        if config["strategy"] == "ucb":
            for _ in range(int(config["online_aux_updates"])):
                lv, _, _, lc = auxiliary_update(vae, vae_optimizer, cost_network, cost_optimizer, replay, config, scaler)
                vae_losses.append(lv); cost_losses.append(lc)

        train_portfolio, train_var, _ = evaluate_strategy(q_network, vae, cost_network, control, config, scaler, train)
        test_portfolio, test_var, _ = evaluate_strategy(q_network, vae, cost_network, control, config, scaler, test, train.iloc[-1])
        train_metrics = period_metrics(train_portfolio, train["time"], train_var)
        test_metrics = period_metrics(test_portfolio, test["time"], test_var)
        for split, values in (("train", train_metrics), ("test", test_metrics)):
            curve_rows.append({"ticker": ticker, "period": period, "model_key": model_key,
                               "model": config["label"], "seed": CURRENT_RUN_SEED,
                               "episode": episode + 1, "split": split,
                               "control": float(control), **values})
        if progress_bar is not None and ((episode + 1) % 5 == 0 or episode + 1 == EPISODES):
            progress_bar.set_postfix(period=period, model=config["strategy"], seed=CURRENT_RUN_SEED,
                                     episode=f"{episode + 1}/{EPISODES}", test_profit=f"{test_metrics['profit']:.2f}")

    train_portfolio, train_var, _ = evaluate_strategy(q_network, vae, cost_network, control, config, scaler, train)
    test_portfolio, test_var, novelty = evaluate_strategy(q_network, vae, cost_network, control, config, scaler, test, train.iloc[-1])
    train_metrics = period_metrics(train_portfolio, train["time"], train_var)
    test_metrics = period_metrics(test_portfolio, test["time"], test_var)
    metric_row = {"ticker": ticker, "period": period, "model_key": model_key,
                  "model": config["label"], "strategy": config["strategy"],
                  "seed": CURRENT_RUN_SEED, "episodes": EPISODES, "gamma": GAMMA,
                  "beta_0": config.get("beta_0", np.nan), "beta_decay": config.get("beta_decay", np.nan),
                  "beta_min": config.get("beta_min", np.nan), "epsilon_init": config.get("epsilon_init", np.nan),
                  "epsilon_decay": config.get("epsilon_decay", np.nan), "epsilon_min": config.get("epsilon_min", np.nan),
                  "vae_latent_dim": config.get("vae_latent_dim", np.nan), "vae_lr": config.get("vae_lr", np.nan),
                  "vae_beta_kl": config.get("vae_beta_kl", np.nan)}
    metric_row.update({f"train_{k}": v for k, v in train_metrics.items()})
    metric_row.update({f"test_{k}": v for k, v in test_metrics.items()})
    metric_row.update({"gap_profit": train_metrics["profit"] - test_metrics["profit"],
                       "gap_roi": train_metrics["roi"] - test_metrics["roi"],
                       "gap_arr": train_metrics["arr"] - test_metrics["arr"],
                       "gap_sharpe": train_metrics["sharpe"] - test_metrics["sharpe"],
                       "q_loss_mean": float(np.mean(q_losses)),
                       "vae_loss_mean": float(np.mean(vae_losses)) if vae_losses else np.nan,
                       "cost_loss_mean": float(np.mean(cost_losses)) if cost_losses else np.nan,
                       "novelty_mean": float(np.mean(novelty)) if len(novelty) else np.nan,
                       "finite": bool(np.isfinite(list(train_metrics.values()) + list(test_metrics.values())).all())})
    loss_rows = []
    for loss_name, values in (("Q loss", q_losses), ("VAE loss", vae_losses), ("Cost loss", cost_losses)):
        for update, value in enumerate(values):
            loss_rows.append({"ticker": ticker, "period": period, "model_key": model_key,
                              "seed": CURRENT_RUN_SEED, "loss_name": loss_name,
                              "update": update, "loss": float(value)})
    dates = [pd.Timestamp(train.iloc[-1]["time"]), *pd.to_datetime(test["time"]).tolist()]
    portfolio_rows = [{"ticker": ticker, "period": period, "model_key": model_key,
                       "seed": CURRENT_RUN_SEED, "step": step, "time": dates[step],
                       "portfolio_value": float(value)}
                      for step, value in enumerate(test_portfolio)]
    checkpoint = {"ticker": ticker, "period": period, "model_key": model_key,
                  "seed": CURRENT_RUN_SEED, "config": config, "metric": metric_row,
                  "scaler_mean": scaler.mean.copy(), "scaler_std": scaler.std.copy(),
                  "q_network_state_dict": q_network.state_dict(),
                  "vae_state_dict": vae.state_dict() if config["strategy"] == "ucb" else None,
                  "cost_network_state_dict": cost_network.state_dict() if config["strategy"] == "ucb" else None}
    del q_network, vae, cost_network, q_optimizer, vae_optimizer, cost_optimizer, replay
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return metric_row, curve_rows, loss_rows, portfolio_rows, checkpoint

def completed_seeds(metrics_path: Path, model_key: str) -> set[int]:
    if not RESUME or not metrics_path.exists(): return set()
    frame = pd.read_csv(metrics_path)
    return set(frame.loc[frame["model_key"] == model_key, "seed"].astype(int)).intersection(SEEDS)

def save_buy_and_hold(period_dir: Path, ticker: str, period: str, train, test):
    metrics_path = period_dir / "buy_and_hold_metrics.csv"
    portfolio_path = period_dir / "buy_and_hold_portfolio.csv"
    portfolio, var_targets, shares, cash = buy_and_hold(train, test)
    metrics = period_metrics(portfolio, test["time"], var_targets)
    pd.DataFrame([{ "ticker": ticker, "period": period, "model_key": "BUY_AND_HOLD",
                    "model": "Buy-and-Hold", "shares": shares, "cash_after_buy": cash,
                    **{f"test_{k}": v for k, v in metrics.items()} }]).to_csv(metrics_path, index=False)
    dates = [pd.Timestamp(train.iloc[-1]["time"]), *pd.to_datetime(test["time"]).tolist()]
    pd.DataFrame({"ticker": ticker, "period": period, "model_key": "BUY_AND_HOLD",
                  "step": np.arange(len(portfolio)), "time": dates,
                  "portfolio_value": portfolio}).to_csv(portfolio_path, index=False)

if RUN_TRAINING:
    total = len(PERIODS) * len(MODEL_KEYS) * len(SEEDS)
    progress = tqdm(total=total, desc=f"{TICKER} — GOOD/BAD — two RL models", unit="seed", dynamic_ncols=True)
    for period in PERIODS:
        train, test = load_period_data(TICKER, period)
        scaler = fit_frozen_scaler(train)
        period_dir = RUN_ROOT / period.lower(); period_dir.mkdir(parents=True, exist_ok=True)
        np.savez(period_dir / "frozen_scaler_train_only.npz", mean=scaler.mean, std=scaler.std)
        save_buy_and_hold(period_dir, TICKER, period, train, test)
        metrics_path = period_dir / "rl_metrics.csv"
        curves_path = period_dir / "rl_episode_curves.csv"
        losses_path = period_dir / "rl_losses.csv"
        portfolios_path = period_dir / "rl_test_portfolios.csv"
        errors_path = period_dir / "errors.csv"
        for model_key in MODEL_KEYS:
            done = completed_seeds(metrics_path, model_key)
            for seed in SEEDS:
                if seed in done:
                    progress.update(1); progress.set_postfix(period=period, model=model_key, seed=seed, status="resume-skip"); continue
                set_seed(seed); started = time.perf_counter()
                try:
                    metric, curves, losses, portfolios, checkpoint = run_one_seed(
                        TICKER, period, model_key, train, test, scaler, progress)
                    metric["elapsed_seconds"] = float(time.perf_counter() - started)
                    maybe_save_xai_checkpoint(checkpoint, metric, metrics_path, model_key, period)
                    append_frame(curves_path, pd.DataFrame(curves))
                    if losses: append_frame(losses_path, pd.DataFrame(losses))
                    append_frame(portfolios_path, pd.DataFrame(portfolios))
                    append_frame(metrics_path, pd.DataFrame([metric]))  # commit cuối cùng cho resume
                except Exception as error:
                    append_frame(errors_path, pd.DataFrame([{"ticker": TICKER, "period": period,
                        "model_key": model_key, "seed": seed, "error_type": type(error).__name__,
                        "error": str(error), "traceback": traceback.format_exc()}]))
                    print(f"FAILED {TICKER} {period} {model_key} seed={seed}: {error}", flush=True)
                finally:
                    progress.update(1); gc.collect()
                    if torch.cuda.is_available(): torch.cuda.empty_cache()
    progress.close()
else:
    print("RUN_TRAINING=False — chỉ đọc CSV và checkpoint XAI hiện có.")


In [ ]:
# CELL 6 — Tổng hợp CSV và 5 biểu đồ cho từng trường hợp GOOD/BAD
COLORS = {"UNCERTAINTY_AWARE_UCB_009": "tab:blue", "EPSILON_GREEDY": "tab:orange", "BUY_AND_HOLD": "tab:green"}
LABELS = {"UNCERTAINTY_AWARE_UCB_009": "Uncertainty-Aware UCB-VAE (vae_019 + ucb_009)",
          "EPSILON_GREEDY": "Deep SARSA Epsilon-Greedy", "BUY_AND_HOLD": "Buy-and-Hold"}

def save_show(fig, path: Path):
    fig.tight_layout(); fig.savefig(path, dpi=220, bbox_inches="tight"); plt.show(); plt.close(fig)

def report_period(ticker: str, period: str):
    period_dir = RUN_ROOT / period.lower(); plots_dir = period_dir / "plots"; plots_dir.mkdir(parents=True, exist_ok=True)
    metrics_path = period_dir / "rl_metrics.csv"
    curves_path = period_dir / "rl_episode_curves.csv"
    portfolios_path = period_dir / "rl_test_portfolios.csv"
    if not metrics_path.exists():
        print(f"Chưa có RL metrics cho {ticker} {period}: {metrics_path}"); return
    metrics = pd.read_csv(metrics_path).drop_duplicates(["model_key", "seed"], keep="last")
    curves = pd.read_csv(curves_path).drop_duplicates(["model_key", "seed", "episode", "split"], keep="last")
    portfolios = pd.read_csv(portfolios_path).drop_duplicates(["model_key", "seed", "step"], keep="last")
    bh_metrics = pd.read_csv(period_dir / "buy_and_hold_metrics.csv")
    bh_portfolio = pd.read_csv(period_dir / "buy_and_hold_portfolio.csv")
    for model_key in MODEL_KEYS:
        present = set(metrics.loc[metrics.model_key == model_key, "seed"].astype(int))
        print(ticker, period, model_key, "completed", len(present), "missing", sorted(set(SEEDS) - present))

    summary_rows = []
    test_columns = ["test_profit", "test_roi", "test_arr", "test_volatility", "test_sharpe", "test_max_drawdown", "test_violations"]
    for model_key in MODEL_KEYS:
        part = metrics[metrics.model_key == model_key]
        row = {"ticker": ticker, "period": period, "model_key": model_key, "model": LABELS[model_key], "runs": part.seed.nunique()}
        for col in test_columns:
            row[f"{col}_mean"] = float(part[col].mean()); row[f"{col}_std"] = float(part[col].std(ddof=1))
        summary_rows.append(row)
    bh = bh_metrics.iloc[0]
    row = {"ticker": ticker, "period": period, "model_key": "BUY_AND_HOLD", "model": LABELS["BUY_AND_HOLD"], "runs": 1}
    for col in test_columns:
        row[f"{col}_mean"] = float(bh[col]); row[f"{col}_std"] = 0.0
    summary_rows.append(row)
    summary = pd.DataFrame(summary_rows)
    summary.to_csv(period_dir / "three_strategy_summary.csv", index=False)
    try: print(summary.to_markdown(index=False, floatfmt=".6f"))
    except ImportError: print(summary.to_string(index=False))

    title_suffix = f"{ticker} {period}"
    # OLD TEST 1 — Learning Curve Comparison: chỉ hai mô hình RL.
    fig, ax = plt.subplots(figsize=(12, 5))
    for model_key in MODEL_KEYS:
        part = curves[(curves.model_key == model_key) & (curves.split == "test")]
        grouped = part.groupby("episode")["profit"].agg(["mean", "std"]).reset_index()
        x = grouped.episode.to_numpy(); y = grouped["mean"].to_numpy(); spread = grouped["std"].fillna(0).to_numpy()
        ax.plot(x, y, label=LABELS[model_key], color=COLORS[model_key], linewidth=2)
        ax.fill_between(x, y - spread, y + spread, color=COLORS[model_key], alpha=0.2)
    ax.set(title=f"Learning Curve Comparison - {title_suffix}", xlabel="Episode", ylabel="Test Profit")
    ax.grid(alpha=0.3); ax.legend()
    save_show(fig, plots_dir / f"{ticker.lower()}_{period.lower()}_learning_curve_comparison.png")

    # OLD TEST 2 — Portfolio Value Comparison: chỉ hai mô hình RL.
    fig, ax = plt.subplots(figsize=(12, 5))
    for model_key in MODEL_KEYS:
        grouped = portfolios[portfolios.model_key == model_key].groupby("step")["portfolio_value"].mean()
        ax.plot(grouped.index, grouped.values, label=LABELS[model_key], color=COLORS[model_key], linewidth=2)
    ax.set(title=f"Portfolio Value Comparison - {title_suffix}", xlabel="Trading Step", ylabel="Portfolio Value")
    ax.grid(alpha=0.3); ax.legend()
    save_show(fig, plots_dir / f"{ticker.lower()}_{period.lower()}_portfolio_value_comparison.png")

    # OLD TEST 3 — Final Profit Across Runs: chỉ hai mô hình RL.
    fig, ax = plt.subplots(figsize=(12, 5))
    for model_key in MODEL_KEYS:
        part = metrics[metrics.model_key == model_key].sort_values("seed")
        ax.plot(part.seed, part.test_profit, marker="o", color=COLORS[model_key], label=f"{LABELS[model_key]} final profit")
        ax.axhline(part.test_profit.mean(), color=COLORS[model_key], linestyle="--", alpha=0.7,
                   label=f"{model_key} mean = {part.test_profit.mean():.2f}")
    ax.set(title=f"Final Profit Across Runs - {title_suffix}", xlabel="Seed", ylabel="Final Profit")
    ax.grid(alpha=0.3); ax.legend()
    save_show(fig, plots_dir / f"{ticker.lower()}_{period.lower()}_final_profit_across_runs.png")

    # NEW 1 — Portfolio Value Comparison của cả ba chiến lược.
    fig, ax = plt.subplots(figsize=(12, 5))
    for model_key in MODEL_KEYS:
        grouped = portfolios[portfolios.model_key == model_key].groupby("step")["portfolio_value"].mean()
        ax.plot(grouped.index, grouped.values, label=LABELS[model_key], color=COLORS[model_key], linewidth=2)
    ax.plot(bh_portfolio.step, bh_portfolio.portfolio_value, label=LABELS["BUY_AND_HOLD"],
            color=COLORS["BUY_AND_HOLD"], linewidth=2)
    ax.axhline(BALANCE_INIT, color="gray", linewidth=1, linestyle=":", label="Initial capital")
    ax.set(title=f"Three-Strategy Portfolio Value Comparison - {title_suffix}",
           xlabel="Trading Step", ylabel="Portfolio Value")
    ax.grid(alpha=0.3); ax.legend()
    save_show(fig, plots_dir / f"{ticker.lower()}_{period.lower()}_three_strategy_portfolio_comparison.png")

    # NEW 2 — Sáu test metrics của cả ba chiến lược.
    metric_specs = [("test_profit", "Final Profit"), ("test_roi", "ROI (%)"),
                    ("test_arr", "ARR (%)"), ("test_sharpe", "Sharpe Ratio"),
                    ("test_volatility", "Volatility (%)"), ("test_max_drawdown", "Max Drawdown (%)")]
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    keys = [*MODEL_KEYS, "BUY_AND_HOLD"]
    short_labels = ["UCB-VAE", "Epsilon-Greedy", "Buy-and-Hold"]
    for ax, (column, title) in zip(axes.ravel(), metric_specs):
        means = [float(summary.loc[summary.model_key == key, f"{column}_mean"].iloc[0]) for key in keys]
        stds = [float(summary.loc[summary.model_key == key, f"{column}_std"].iloc[0]) for key in keys]
        ax.bar(short_labels, means, yerr=stds, capsize=5, color=[COLORS[key] for key in keys])
        ax.axhline(0, color="black", linewidth=0.8); ax.set_title(title); ax.grid(axis="y", alpha=0.25)
        ax.tick_params(axis="x", rotation=12)
    fig.suptitle(f"Three-Strategy Final Test Metrics - {title_suffix}", fontsize=15)
    save_show(fig, plots_dir / f"{ticker.lower()}_{period.lower()}_three_strategy_final_metrics.png")

for period in PERIODS:
    report_period(TICKER, period)


In [ ]:
# CELL 7 — Kiểm tra nhanh cấu hình và vị trí output
assert tuple(range(41, 61)) == SEEDS and len(SEEDS) == 20
assert XAI_MODEL_KEY in MODEL_KEYS and XAI_PERIOD in PERIODS
assert XAI_SELECTION_METRIC.startswith("train_")
assert MODEL_CONFIGS["UNCERTAINTY_AWARE_UCB_009"]["vae_latent_dim"] == 16
assert MODEL_CONFIGS["UNCERTAINTY_AWARE_UCB_009"]["vae_lr"] == 1e-4
assert MODEL_CONFIGS["UNCERTAINTY_AWARE_UCB_009"]["vae_beta_kl"] == 0.05
assert MODEL_CONFIGS["UNCERTAINTY_AWARE_UCB_009"]["beta_0"] == 0.03
assert MODEL_CONFIGS["UNCERTAINTY_AWARE_UCB_009"]["beta_decay"] == 0.90
assert MODEL_CONFIGS["UNCERTAINTY_AWARE_UCB_009"]["beta_min"] == 0.01
assert MODEL_CONFIGS["EPSILON_GREEDY"]["epsilon_init"] == 1.0
assert MODEL_CONFIGS["EPSILON_GREEDY"]["epsilon_decay"] == 0.95
assert MODEL_CONFIGS["EPSILON_GREEDY"]["epsilon_min"] == 0.05
for period in PERIODS:
    train, test = load_period_data(TICKER, period)
    scaler = fit_frozen_scaler(train)
    assert scaler.mean.shape == (7,) and scaler.std.shape == (7,)
    assert np.isfinite(scaler.mean).all() and np.isfinite(scaler.std).all() and (scaler.std > 0).all()
print("CONFIG_AND_DATA_SMOKE_TEST_OK")
print("Kết quả của máy này:", RUN_ROOT)
